# 03 - Non-Maximum Suppression from scratch

**Question:** a detector proposes many boxes for one object. How do we keep one per object, and how does the overlap threshold change the result?

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
FIG = os.path.abspath("../figures"); RES = os.path.abspath("../results")
os.makedirs(FIG, exist_ok=True); os.makedirs(RES, exist_ok=True)
VOC = os.path.abspath("../data/VOCdevkit/VOC2007")
from src.iou import calculate_iou, iou_matrix
from src.nms import confidence_filter, non_max_suppression_simple, non_max_suppression, class_aware_nms
from src.visualization import draw_boxes, show_image
from src.dataset import VOC_CLASSES

## 1. Concept

A detector predicts at many grid positions, so neighbouring positions all fire on the same object. Two different filters clean this up and they are easy to mix up:

| filter | question it asks | parameter |
|---|---|---|
| confidence filter | is this single prediction confident enough? | confidence threshold |
| NMS | does this box overlap an already chosen, better box too much? | IoU threshold |

Algorithm (greedy):

```
sort boxes by score, highest first
while boxes remain:
    take the best box -> keep it
    remove every remaining box whose IoU with it is > iou_threshold
```

Direction to remember: **lower IoU threshold = more suppression** (boxes only need a little overlap to be removed); higher threshold = more duplicates survive.

NMS is run **per class**, otherwise a "person" box could delete an overlapping "chair" box.

## 2. Worked example by hand

Three predictions, as in the guide: scores 0.95, 0.91, 0.40. To make the threshold matter, B and C are copies of A shifted sideways by 30 and 50 pixels.

In [2]:
boxes = np.array([[0, 0, 100, 100], [30, 0, 130, 100], [50, 0, 150, 100.]])      # A, B, C
scores = np.array([0.95, 0.91, 0.40]); names = "ABC"
M = iou_matrix(boxes, boxes)
print(pd.DataFrame(M, index=list(names), columns=list(names)).round(3))
print("\nA vs B = 7000/13000 = %.3f ; A vs C = 5000/15000 = %.3f ; B vs C = %.3f" % (M[0,1], M[0,2], M[1,2]))

       A      B      C
A  1.000  0.538  0.333
B  0.538  1.000  0.667
C  0.333  0.667  1.000

A vs B = 7000/13000 = 0.538 ; A vs C = 5000/15000 = 0.333 ; B vs C = 0.667


### Trace of the algorithm at IoU threshold 0.5

In [3]:
order = list(np.argsort(-scores)); keep = []; log = []
while order:
    best = order.pop(0); keep.append(best); removed = []
    for j in list(order):
        iou = calculate_iou(boxes[best], boxes[j])
        if iou > 0.5: order.remove(j); removed.append(f"{names[j]} (IoU {iou:.2f} > 0.5)")
    log.append((names[best], "kept", ", ".join(removed) or "nothing removed", "".join(names[k] for k in order) or "-"))
display(pd.DataFrame(log, columns=["step: best box", "decision", "removed because of overlap", "still waiting"]))
print("final:", [names[k] for k in keep])

,step: best box,decision,removed because of overlap,still waiting
0,A,kept,B (IoU 0.54 > 0.5),C
1,C,kept,nothing removed,-


final: ['A', 'C']


### Three thresholds on the same boxes

In [4]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.2)); rows = []
for ax, (title, thr) in zip(axes, [("before NMS", None), ("IoU thr 0.3", 0.3), ("IoU thr 0.5", 0.5), ("IoU thr 0.7", 0.7)]):
    kept = list(range(3)) if thr is None else non_max_suppression(boxes, scores, thr)
    ax.set_xlim(-10, 180); ax.set_ylim(140, -20); ax.set_aspect("equal")
    for k in range(3):
        draw_boxes(ax, [boxes[k]], color="green" if k in kept else "lightgray", lw=3 if k in kept else 1.5, linestyle="-" if k in kept else "--")
        ax.text(boxes[k][0] + 3, boxes[k][1] + 14 + 12 * k, f"{names[k]} {scores[k]:.2f}", fontsize=9)
    ax.set_title(title + ("" if thr is None else f"  -> keeps {''.join(names[k] for k in kept)}"))
    if thr: rows.append((thr, "".join(names[k] for k in kept)))
plt.tight_layout(); plt.savefig(f"{FIG}/nms_comparison.png", dpi=110); plt.show()
print(rows)
assert non_max_suppression(boxes, scores, 0.3) == [0] and non_max_suppression(boxes, scores, 0.5) == [0, 2] and non_max_suppression(boxes, scores, 0.7) == [0, 1, 2]

[(0.3, 'A'), (0.5, 'AC'), (0.7, 'ABC')]


## 3. Tests

The readable loop version and the vectorised version must give identical answers; class-aware NMS must not let one class suppress another.

In [5]:
import time
rng = np.random.RandomState(1)
xy = rng.rand(300, 2) * 100; wh = rng.rand(300, 2) * 40 + 5
B = np.concatenate([xy, xy + wh], 1); S = rng.rand(300)
for t in (0.3, 0.5, 0.7):
    assert non_max_suppression(B, S, t) == non_max_suppression_simple(B, S, t)
print("simple == vectorised for 300 random boxes at 0.3, 0.5, 0.7")
two = np.array([[0, 0, 10, 10], [0, 0, 10, 10.]])
assert len(class_aware_nms(two, [0.9, 0.8], [0, 1], 0.5)) == 2 and len(class_aware_nms(two, [0.9, 0.8], [0, 0], 0.5)) == 1
b, s = confidence_filter(boxes, scores, 0.5); assert len(b) == 2
print("class-aware and confidence filter tests passed")

simple == vectorised for 300 random boxes at 0.3, 0.5, 0.7
class-aware and confidence filter tests passed


## 4. NMS on real detector output

Raw YOLOv8n output at 640 px, confidence >= 0.25: the image with the most candidate boxes, before and after NMS.

In [6]:
from src.experiments import load_gts, load_candidates, detections
from src.dataset import load_voc_subset
ids, gts = load_gts(); cands = load_candidates("clean_640")
_, _, paths = load_voc_subset(VOC, 1000, 0)
n_cand = {i: int((cands[i]["scores"] >= 0.25).sum()) for i in ids}
i = max(n_cand, key=n_cand.get)
c = cands[i]; k = c["scores"] >= 0.25
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
show_image(axes[0], paths[i]); draw_boxes(axes[0], c["boxes"][k], c["labels"][k], c["scores"][k], color="red", lw=1)
axes[0].set_title(f"{i}: {k.sum()} candidates before NMS")
for ax, thr in zip(axes[1:], (0.3, 0.7)):
    d = detections({i: c}, 0.25, thr)[i]; show_image(ax, paths[i]); draw_boxes(ax, d["boxes"], d["labels"], d["scores"], color="lime", lw=2)
    ax.set_title(f"after NMS (IoU thr {thr}): {len(d['scores'])} boxes")
plt.tight_layout(); plt.savefig(f"{FIG}/nms_real_image.png", dpi=100); plt.show()

### Over the whole subset: how many boxes survive?

In [7]:
rows = []
for thr in (0.3, 0.5, 0.7, None):
    d = detections(cands, 0.25, thr)
    rows.append(dict(nms_iou="off" if thr is None else thr, boxes_per_image=np.mean([len(v["scores"]) for v in d.values()]),
                     gt_per_image=np.mean([(~g["difficult"]).sum() for g in gts.values()])))
df = pd.DataFrame(rows); display(df.round(3))

,nms_iou,boxes_per_image,gt_per_image
0,0.3,3.030,2.422
1,0.5,3.132,2.422
2,0.7,3.330,2.422
3,off,25.019,2.422


### A crowded scene: NMS can remove correct neighbours

Images with at least 4 people. Same detections, three NMS thresholds.

In [8]:
crowd = [i for i in ids if ((gts[i]["labels"] == 14) & ~gts[i]["difficult"]).sum() >= 4]
j = max(crowd, key=lambda x: (gts[x]["labels"] == 14).sum()); print(len(crowd), "images with >= 4 people; example", j, "has", (gts[j]["labels"] == 14).sum(), "people")
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, thr in zip(axes, (0.3, 0.5, 0.7)):
    d = detections({j: cands[j]}, 0.25, thr)[j]; p = d["labels"] == 14
    show_image(ax, paths[j]); draw_boxes(ax, gts[j]["boxes"][gts[j]["labels"] == 14], color="lime", lw=2)
    draw_boxes(ax, d["boxes"][p], color="red", lw=1.5, linestyle="--"); ax.set_title(f"NMS IoU {thr}: {p.sum()} person boxes (green = GT)")
plt.tight_layout(); plt.savefig(f"{FIG}/nms_crowded_scene.png", dpi=100); plt.show()

63 images with >= 4 people; example 006693 has 17 people


### Class-aware vs class-agnostic NMS (whole subset)

In [9]:
from src.evaluation import Evaluator, apply_pipeline
agn = lambda b, s, l, t: class_aware_nms(b, s, np.zeros(len(l), int), t)
rows = []
for name, fn in (("class-aware", class_aware_nms), ("class-agnostic", agn)):
    m = Evaluator(gts, apply_pipeline(cands, 0.25, 0.5, fn)).metrics(0.5)
    rows.append(dict(NMS=name, detections=m["n_det"], TP=m["tp"], FP=m["fp"], FN=m["fn"], precision=round(m["precision"], 4), recall=round(m["recall"], 4)))
display(pd.DataFrame(rows))

,NMS,detections,TP,FP,FN,precision,recall
0,class-aware,2853,2027,826,395,0.7105,0.8369
1,class-agnostic,2780,2000,780,422,0.7194,0.8258


## 5. Observation

* In the hand example IoU(A,B) = 0.54 and IoU(A,C) = 0.33. Threshold 0.3 keeps only A, threshold 0.5 keeps A and C (B is removed), threshold 0.7 keeps all three. C's low score (0.40) plays no role in NMS; only a confidence filter would remove it.
* On a real image dozens of candidate boxes collapse to a handful (figure `nms_real_image.png`), and the table in section 4 shows how many boxes per image remain against the number of true objects per image.
* In the crowded scene a low threshold (0.3) merges neighbouring people into one box; a high threshold keeps more boxes but also leaves more duplicates. There is no universally right value, which is why this is tested as an experiment in notebook 06.
* Class-agnostic NMS lets a box of one class remove an overlapping box of another class; the last table shows what that costs on this data.